In [1]:
import cobra
import pandas as pd
import numpy as np
from scipy.optimize import linprog
from scipy.stats import ttest_ind

from cobra.flux_analysis import pfba, flux_variability_analysis
from cobra.manipulation.delete import (
    remove_genes,
    prune_unused_metabolites,
    prune_unused_reactions,
)
from cobra.sampling import sample

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# Load the SBML model
model = cobra.io.read_sbml_model(
    "/Users/karthik/Desktop/PHCCO IISc Internship/EMT/Recon3D.xml"
)
model.reactions

[<Reaction 24_25DHVITD3tm at 0x32cb3f9d0>,
 <Reaction 25HVITD3t at 0x32cb3f950>,
 <Reaction COAtl at 0x32cb40290>,
 <Reaction EX_5adtststerone_e at 0x32cb407d0>,
 <Reaction EX_5adtststerones_e at 0x32cb41450>,
 <Reaction EX_5fthf_e at 0x32cb41790>,
 <Reaction EX_5htrp_e at 0x32cb41b10>,
 <Reaction EX_5mthf_e at 0x32cb41f90>,
 <Reaction EX_5thf_e at 0x32cb42510>,
 <Reaction EX_6dhf_e at 0x32cb42d10>,
 <Reaction 24_25VITD3Hm at 0x32cb430d0>,
 <Reaction 24NPHte at 0x32cb434d0>,
 <Reaction 10FTHF7GLUtl at 0x32cb4c650>,
 <Reaction 10FTHFtm at 0x32cb4c510>,
 <Reaction 11DOCRTSLtr at 0x32cb4d1d0>,
 <Reaction 13DAMPPOX at 0x32cb4d750>,
 <Reaction 24_25DHVITD2t at 0x32cb4dc90>,
 <Reaction 24_25DHVITD2tm at 0x32cb4fe10>,
 <Reaction 24_25DHVITD3t at 0x32cb4ed90>,
 <Reaction 25VITD2Hm at 0x32cb50150>,
 <Reaction 2AMACHYD at 0x32cb50650>,
 <Reaction 2AMACSULT at 0x32cb4edd0>,
 <Reaction 2AMADPTm at 0x32cb515d0>,
 <Reaction 2MCITt at 0x32cb51590>,
 <Reaction 2OXOADOXm at 0x32cb528d0>,
 <Reaction 2OX

In [4]:
# Display all reaction IDs that might indicate biomass reaction
biomass_reactions = [rxn.id for rxn in model.reactions if "biomass" in rxn.id.lower()]
print("Potential biomass reactions:", biomass_reactions)

# Access the biomass reaction by its ID (example: 'biomass_reaction')
# Adjust the ID based on the actual ID in the Recon3D model
biomass_reaction_id = biomass_reactions[
    0
]  # Select the first one or the correct one from the list
biomass_reaction = model.reactions.get_by_id(biomass_reaction_id)

# Display the biomass reaction
print("Biomass reaction:", biomass_reaction)

# You can also display the reaction formula
print("Biomass reaction formula:", biomass_reaction.reaction)

Potential biomass reactions: ['BIOMASS_reaction', 'BIOMASS_maintenance', 'BIOMASS_maintenance_noTrTr']
Biomass reaction: BIOMASS_reaction: 0.505626 ala__L_c + 0.35926 arg__L_c + 0.279425 asn__L_c + 0.352607 asp__L_c + 20.704451 atp_c + 0.020401 chsterol_c + 0.011658 clpn_hs_c + 0.039036 ctp_c + 0.046571 cys__L_c + 0.013183 datp_n + 0.009442 dctp_n + 0.009898 dgtp_n + 0.013091 dttp_n + 0.275194 g6p_c + 0.325996 gln__L_c + 0.385872 glu__L_c + 0.538891 gly_c + 0.036117 gtp_c + 20.650823 h2o_c + 0.126406 his__L_c + 0.286078 ile__L_c + 0.545544 leu__L_c + 0.592114 lys__L_c + 0.153018 met__L_c + 0.023315 pail_hs_c + 0.154463 pchol_hs_c + 0.055374 pe_hs_c + 0.002914 pglyc_hs_c + 0.259466 phe__L_c + 0.412484 pro__L_c + 0.005829 ps_hs_c + 0.392525 ser__L_c + 0.017486 sphmyln_hs_c + 0.31269 thr__L_c + 0.013306 trp__L_c + 0.159671 tyr__L_c + 0.053446 utp_c + 0.352607 val__L_c --> 20.650823 adp_c + 20.650823 h_c + 20.650823 pi_c
Biomass reaction formula: 0.505626 ala__L_c + 0.35926 arg__L_c + 0.27

In [5]:
biomass_reaction = {
    "ala_L_c": 0.606,
    "arg_L_c": 0.391,
    "asn_L_c": 0.333,
    "asp_L_c": 0.269,
    "cys_L_c": 0.163,
    "gln_L_c": 0.328,
    "glu_L_c": 0.399,
    "gly_c": 0.519,
    "his_L_c": 0.142,
    "ile_L_c": 0.329,
    "leu_L_c": 0.565,
    "lys_L_c": 0.571,
    "met_L_c": 0.133,
    "phe_L_c": 0.176,
    "pro_L_c": 0.245,
    "ser_L_c": 0.397,
    "thr_L_c": 0.389,
    "trp_L_c": 0.0412,
    "tyr_L_c": 0.155,
    "val_L_c": 0.397,
    "damp_c": 0.0136,
    "dcmp_c": 0.00907,
    "dgmp_c": 0.00907,
    "dtmp_c": 0.0136,
    "cmp_c": 0.0301,
    "gmp_c": 0.0503,
    "ump_c": 0.057,
    "amp_c": 0.0301,
    "glygn2_c": 0.341,
    "sphmyln_hs_c": 0.00918,
    "chsterol_c": 0.0262,
    "xolest_hs_c": 0.0223,
    "mag_hs_c": 0.00377,
    "dag_hs_c": 0.00377,
    "pail_hs_c": 0.00983,
    "pe_hs_c": 0.0442,
    "ps_hs_c": 0.0151,
    "pchol_hs_c": 0.0472,
    "lpchol_hs_c": 0.00328,
    "clpn_hs_c": 0.00197,
    "pa_hs_c": 0.0302,
    "hdcea_c": 0.00298,
    "hdca_c": 0.00745,
    "ocdcea_c": 0.0252,
    "ocdca_c": 0.00447,
    "ptrc_c": 0.0014,
    "spmd_c": 0.016,
    "sprm_c": 0.006,
    "gthrd_c": 0.01,
    "nad_c": 0.003,
    "nadp_c": 0.0003,
    "Q10_c": 0.0000486,
    "paps_c": 0.0025,
    "thbpt_c": 0.00007,
    "crn_c": 0.02,
    "atp_c": 0.003,
    "adp_c": 0.003,
    "pi_c": 0.003,
    "h2o_c": 0.003,
    "h_c": 0.003,
}

In [3]:
# Check for essential metabolites and gaps in the model
summary = model.summary()
print(summary)

Objective
1.0 BIOMASS_maintenance = 755.003215550663

Uptake
------
  Metabolite        Reaction  Flux  C-Number C-Flux
     thmtp_c      DM_thmtp_c  1000        12  4.09%
  12ppd__R_e   EX_12ppd__R_e 250.8         3  0.26%
      5aop_e       EX_5aop_e  1000         5  1.70%
   HC00250_e    EX_HC00250_e  1000         0  0.00%
   HC00900_e    EX_HC00900_e  1000         4  1.36%
   HC01361_e    EX_HC01361_e  1000         9  3.07%
     Lkynr_e      EX_Lkynr_e 105.1        10  0.36%
   acetone_e    EX_acetone_e 427.7         3  0.44%
     alltn_e      EX_alltn_e  1000         4  1.36%
       atp_e        EX_atp_e  1000        10  3.41%
       cit_e        EX_cit_e 718.8         6  1.47%
    crm_hs_e     EX_crm_hs_e  1000        19  6.47%
      dcmp_e       EX_dcmp_e  1000         9  3.07%
      dopa_e       EX_dopa_e  1000         8  2.73%
      dtmp_e       EX_dtmp_e  1000        10  3.41%
gluside_hs_e EX_gluside_hs_e  13.2        25  0.11%
      h2o2_e       EX_h2o2_e  1000         0  0.

In [ ]:
for i, reaction in enumerate(model.reactions):
    print(f"{i}. ID: {reaction.id} Rxn: {reaction.name}")

In [ ]:
# Remove any unused metabolites or reactions
cobra.manipulation.delete.prune_unused_metabolites(model)

In [ ]:
print(model.objective)

In [ ]:
solution = model.optimize()
solution.objective_value

## Processing the expression file `s11.xlsx`:


In [ ]:
# Load the Excel file
file_path = "/Users/karthik/Desktop/PHCCO IISc Internship/EMT/s11.xlsx"
df = pd.read_excel(file_path)
df

In [ ]:
# Rename columns to desired names
df.columns = [
    "symbol",
    "baseMean",
    "log2fold_change",
    "lfcSE",
    "stat",
    "p_value",
    "padj",
]

# Filter columns and drop rows with missing values
df = df[["symbol", "log2fold_change", "p_value"]].dropna()

# Calculate -log10(p-value)
df["-log10(p_value)"] = -np.log10(df["p_value"])

# Set thresholds for classification
log2_fold_change_up_threshold = 1
log2_fold_change_down_threshold = -1
p_value_threshold = 0.05

# Define a column to indicate upregulated, downregulated, and not significant
df["regulation"] = np.where(
    (df["log2fold_change"] >= log2_fold_change_up_threshold)
    & (df["p_value"] < p_value_threshold),
    "Upregulated",
    np.where(
        (df["log2fold_change"] <= log2_fold_change_down_threshold)
        & (df["p_value"] < p_value_threshold),
        "Downregulated",
        "Not significant",
    ),
)

print(df)

In [ ]:
df.to_csv("processed_s11_expression_data.csv", index=False)

In [ ]:
df = pd.read_csv("processed_s11_expression_data.csv")

In [ ]:
# Initialize the figure
plt.figure(figsize=(12, 8))

# Create the volcano plot
sns.scatterplot(
    data=df,
    x="log2fold_change",
    y="-log10(p_value)",
    hue="regulation",
    palette={"Upregulated": "red", "Downregulated": "blue", "Not significant": "grey"},
    edgecolor=None,
    alpha=0.7,
)

# Add vertical lines for log2 fold change thresholds
plt.axvline(x=log2_fold_change_up_threshold, color="grey", linestyle="--")
plt.axvline(x=log2_fold_change_down_threshold, color="grey", linestyle="--")

# Add a horizontal line for the p-value threshold
plt.axhline(y=-np.log10(p_value_threshold), color="grey", linestyle="--")

# Customize the plot
plt.xlabel("log2 Fold Change", fontsize=14)
plt.ylabel("-log10(p-value)", fontsize=14)
plt.title("Compute metrics of the Expression Dataset (S11 Breast Cancer)", fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)

# Annotate points with gene symbols
for i, row in df.iterrows():
    if row["-log10(p_value)"] > 9.1 or abs(row["log2fold_change"]) > 2:
        plt.annotate(
            row["symbol"],
            (row["log2fold_change"], row["-log10(p_value)"]),
            fontsize=9,
            alpha=0.9,
        )

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
# Load the expression data
# expression_data = pd.read_excel(
#     "/Users/karthik/Desktop/PHCCO IISc Internship/EMT/example.xlsx",
#     sheet_name="Sheet1", header=None)

expression_data = pd.read_csv(
    "/Users/karthik/Desktop/PHCCO IISc Internship/EMT/processed_s11_expression_data.csv"
)

# Filter to keep only 'upregulated' and 'downregulated' rows
filtered_data = expression_data[
    expression_data["regulation"].isin(["Upregulated", "Downregulated"])
]

# Extract the relevant columns
filtered_data = filtered_data[["symbol", "log2fold_change", "regulation"]]

# Convert to a dictionary with 'uniprot_id' as the key and 'log2fold_change' as the value
filtered_dict = filtered_data.set_index("symbol")["log2fold_change"].to_dict()

# Print or save the filtered upregulated data
print("Filtered Upregulated Data:")
filtered_data

In [ ]:
# Optionally, save to a new CSV file
filtered_data.to_csv(
    "/Users/karthik/Desktop/PHCCO IISc Internship/EMT/filtered_processed_s11_expression.csv",
    index=False,
)

In [ ]:
filtered_data = pd.read_csv(
    "/Users/karthik/Desktop/PHCCO IISc Internship/EMT/filtered_processed_s11_expression.csv"
)

In [ ]:
genes = []

for gene_id in model.genes:
    genes.append(gene_id.id)

genes

In [ ]:
# Keep only reactions in the model that have associated genes in your expression data
genes_in_filtered_expression_data = filtered_data.copy()["symbol"].to_list()

genes_to_keep = []
for gene_id in model.genes:
    if gene_id.name in genes_in_filtered_expression_data:
        genes_to_keep.append(gene_id.id)

In [ ]:
# Test code on how to iterate over the structure of the model
rxn_ids = model.genes.get_by_id("26_AT1").reactions
rxn_ids = [rxn.id for rxn in rxn_ids]  # Extracting reaction IDs as strings

print("13DAMPPOX" in rxn_ids)

In [ ]:
def remove_genes_and_reactions(model, genes_in_filtered_processed_expression_data):

    genes_to_remove = set()
    for gene_id in model.genes:
        if gene_id.name not in genes_in_filtered_processed_expression_data:
            genes_to_remove.add(gene_id.id)

    print(f"Genes to Remove: {genes_to_remove}")

    remove_genes(model, genes_to_remove, remove_reactions=True)
    prune_unused_metabolites(model)
    prune_unused_reactions(model)

    return model


base_model = remove_genes_and_reactions(
    model, genes_in_filtered_processed_expression_data=genes_in_filtered_expression_data
)
base_model

In [ ]:
cobra.io.write_sbml_model(base_model, "base_model.xml")

In [ ]:
base_model = cobra.io.read_sbml_model("base_model.xml")
base_model

In [ ]:
flux_map(
    base_model,
    display_name_format=lambda x: str(x.id),
    figsize=(300, 250),
    flux_dict={rxn.id: None for rxn in base_model.reactions},
)

In [ ]:
solution = base_model.optimize()
solution

In [ ]:
# Sampling the model
sample_size = 1000
samples = sample(base_model, n=sample_size, processes=6)

samples_df = pd.DataFrame(samples)
samples_df

In [ ]:
def map_genes_to_reactions(model, expression_dict):
    reaction_scores = []

    for gene in model.genes:
        genes_reactions = gene.reactions
        expressions = expression_dict.get(gene.name, 0)

        for reaction in genes_reactions:
            if isinstance(expressions, list):
                if reaction.name in [entry["Reaction"] for entry in reaction_scores]:
                    # Average the expressions in case of multiple scores
                    existing_entry = next(
                        entry
                        for entry in reaction_scores
                        if entry["Reaction"] == reaction.name
                    )
                    existing_entry["Score"] = (
                        existing_entry["Score"] + sum(expressions) / len(expressions)
                    ) / 2
                else:
                    reaction_scores.append(
                        {
                            "Gene": gene.name,
                            "ReactionID": reaction.id,
                            "Reaction": reaction.name,
                            "Score": sum(expressions) / len(expressions),
                        }
                    )
            else:
                if reaction.name in [entry["Reaction"] for entry in reaction_scores]:
                    existing_entry = next(
                        entry
                        for entry in reaction_scores
                        if entry["Reaction"] == reaction.name
                    )
                    existing_entry["Score"] = (
                        existing_entry["Score"] + expressions
                    ) / 2
                else:
                    reaction_scores.append(
                        {
                            "Gene": gene.name,
                            "ReactionID": reaction.id,
                            "Reaction": reaction.name,
                            "Score": expressions,
                        }
                    )

    return reaction_scores


reaction_scores = map_genes_to_reactions(base_model, filtered_dict)
reaction_scores_df = pd.DataFrame(reaction_scores)
reaction_scores_df

In [ ]:
# Sort reaction_scores_df by 'Score' in descending order
reaction_scores_df = reaction_scores_df.sort_values(by="Score", ascending=False)

plt.figure(figsize=(12, 10))
sns.barplot(x="Score", y="Gene", data=reaction_scores_df)
plt.xlabel("Score")
plt.ylabel("Genes that are responsible for Reaction")
plt.title("Causal Genes for the following Reaction Scores in Base Model")
plt.tight_layout()
plt.show()

In [ ]:
# Sort reactions by score
reaction_scores_df = reaction_scores_df.sort_values(by="Score", ascending=False)

# Plotting vertical violin plot
plt.figure(figsize=(10, 8))
sns.violinplot(x="Score", data=reaction_scores_df, inner="point", linewidth=1)
plt.xlabel("Scores")
plt.ylabel("Reactions")
plt.title("Distribution of Reaction Scores in Base Model")
plt.tight_layout()
plt.show()

### iMAT algorithm (Legit):


In [ ]:
def create_context_specific_model(
    general_model_file,
    gene_expression_file,
    recon_algorithm,
    objective,
    exclude_rxns,
    force_rxns,
    bound_rxns,
    bound_lb,
    bound_ub,
    solver,
    context_name,
    low_thresh,
    high_thresh,
):
    """
    Seed a context specific model. Core reactions are determined from GPR associations with gene expression logicals.
    Core reactions that do not necessarily meet GPR association requirements can be forced if in the force reaction
    file. Metabolite exchange (media), sinks, and demands are determined from exchanges file. Reactions can also be
    force excluded even if they meet GPR association requirements using the force exclude file.
    """
    if general_model_file[-4:] == ".mat":
        cobra_model = cobra.io.load_matlab_model(general_model_file)
    elif general_model_file[-4:] == ".xml":
        cobra_model = cobra.io.load_sbml_model(general_model_file)
    elif general_model_file[-5:] == ".json":
        cobra_model = cobra.io.load_json_model(general_model_file)
    else:
        raise NameError("reference model format must be .xml, .mat, or .json")

    cobra_model.objective = {
        getattr(cobra_model.reactions, objective): 1
    }  # set objective

    if objective not in force_rxns:
        force_rxns.append(objective)

    # set boundaries
    cobra_model, bound_rm_rxns = set_boundaries(
        cobra_model, bound_rxns, bound_lb, bound_ub
    )

    # set solver
    cobra_model.solver = solver.lower()

    # check number of unsolvable reactions for reference model under media assumptions
    # incon_rxns, cobra_model = feasibility_test(cobra_model, "before_seeding")
    incon_rxns = []

    # CoBAMP methods
    cobamp_model = COBRAModelObjectReader(
        cobra_model
    )  # load model in readable format for CoBAMP
    cobamp_model.get_irreversibilities(True)
    s_matrix = cobamp_model.get_stoichiometric_matrix()
    lb, ub = cobamp_model.get_model_bounds(False, True)
    rx_names = cobamp_model.get_reaction_and_metabolite_ids()[0]

    # get expressed reactions
    expression_rxns, expr_vector = map_expression_to_rxn(
        cobra_model,
        gene_expression_file,
        recon_algorithm,
        high_thresh=high_thresh,
        low_thresh=low_thresh,
    )

    # find reactions in the force reactions file that are not in general model and warn user
    for rxn in force_rxns:
        if rxn not in rx_names:
            warn(
                f"{rxn} from force reactions not in general model, check BiGG (or relevant database used in your "
                "general model) for synonyms"
            )

    # collect list of reactions that are infeasible but active in expression data or user defined
    infeas_exp_rxns = []
    infeas_force_rxns = []
    infeas_exp_cnt = 0
    infeas_force_cnt = 0

    for idx, (rxn, exp) in enumerate(expression_rxns.items()):
        # log reactions in expressed and force lists that are infeasible that the user may wish to review
        if rxn in incon_rxns and expr_vector[idx] == 1:
            infeas_exp_cnt += 1
            infeas_exp_rxns.append(rxn)
        if rxn in incon_rxns and rxn in force_rxns:
            infeas_force_cnt += 1
            infeas_force_rxns.append(rxn)

        # make changes to expressed reactions base on user defined force/exclude reactions
        # TODO: if not using bound reactions file, add two sets of exchange reactoins to be put in either low or mid bin

        if rxn in force_rxns:
            expr_vector[idx] = (
                high_thresh + 0.1 if recon_algorithm in ["TINIT", "IMAT"] else 1
            )
        if rxn in incon_rxns or rxn in exclude_rxns:
            expr_vector[idx] = (
                low_thresh - 0.1 if recon_algorithm in ["TINIT", "IMAT"] else 0
            )

    idx_obj = rx_names.index(objective)
    idx_force = [rx_names.index(rxn) for rxn in force_rxns if rxn in rx_names]
    exp_idx_list = [idx for (idx, val) in enumerate(expr_vector) if val > 0]
    exp_thresh = (low_thresh, high_thresh)

    # switch case dictionary runs the functions making it too slow, better solution then elif ladder?
    if recon_algorithm == "GIMME":
        context_model_cobra = seed_gimme(
            cobra_model, s_matrix, lb, ub, idx_obj, expr_vector
        )
    elif recon_algorithm == "FASTCORE":
        context_model_cobra = seed_fastcore(
            cobra_model, s_matrix, lb, ub, exp_idx_list, solver
        )
    elif recon_algorithm == "IMAT":
        flux_df: pd.DataFrame
        context_model_cobra: cobra.Model
        context_model_cobra, flux_df = seed_imat(
            cobra_model,
            s_matrix,
            lb,
            ub,
            expr_vector,
            exp_thresh,
            idx_force,
            context_name,
        )
        imat_reactions = flux_df.rxn
        model_reactions = [reaction.id for reaction in context_model_cobra.reactions]
        reaction_intersections = set(imat_reactions).intersection(model_reactions)
        flux_df = flux_df[~flux_df["rxn"].isin(reaction_intersections)]
        flux_df.to_csv(
            str(
                os.path.join(
                    configs.data_dir,
                    "results",
                    context_name,
                    f"{recon_algorithm}_flux.csv",
                )
            )
        )

    elif recon_algorithm == "TINIT":
        context_model_cobra = seed_tinit(
            cobra_model, s_matrix, lb, ub, expr_vector, solver, idx_force
        )
    else:
        raise ValueError(
            f"Unsupported reconstruction algorithm: {recon_algorithm}. Must be 'IMAT', 'GIMME', or 'FASTCORE'"
        )

    # check number of unsolvable reactions for seeded model under media assumptions
    # incon_rxns_cs, context_model_cobra = feasibility_test(context_model_cobra, "after_seeding")
    incon_rxns_cs = []

    # if recon_algorithm in ["IMAT"]:
    #     final_rxns = [rxn.id for rxn in context_model_cobra.reactions]
    #     imat_rxns = flux_df.rxn
    #     for rxn in imat_rxns:
    #         if rxn not in final_rxns:
    #             flux_df = flux_df[flux_df.rxn != rxn]
    #
    #     flux_df.to_csv(os.path.join(configs.datadir, "results", context_name, f"{recon_algorithm}_flux.csv"))

    incon_df = pd.DataFrame({"general_infeasible_rxns": list(incon_rxns)})
    infeas_exp_df = pd.DataFrame({"expressed_infeasible_rxns": infeas_exp_rxns})
    infeas_force_df = pd.DataFrame({"infeasible_rxns_in_force_list": infeas_exp_rxns})
    incon_df_cs = pd.DataFrame({"infeasible_rxns_from_seeding": list(incon_rxns_cs)})
    infeasible_df = pd.concat(
        [incon_df, infeas_exp_df, infeas_force_df, incon_df_cs],
        ignore_index=True,
        axis=1,
    )
    infeasible_df.columns = [
        "InfeasRxns",
        "ExpressedInfeasRxns",
        "ForceInfeasRxns",
        "ContextInfeasRxns",
    ]

    return context_model_cobra, exp_idx_list, infeasible_df

In [ ]:
# high_threshold = expression_data['expression_data'].quantile(0.75)
# low_threshold = expression_data['expression_data'].quantile(0.25)

# Set high and low expression thresholds based on mean and standard deviation
# High expression = top 10%

# Apply iMAT
# NOTE: iMAT with IQR for calculating threshold gives number of high expressed genes, around 100 among ~600
# NOTE: iMAT with mean + or - std threshold gives number of high expressed genes: 2 out of ~600!! So i rejected it.


def apply_imat(model, reaction_scores_df, expression_data):

    high_threshold = expression_data["log2fold_change"].quantile(0.75)
    low_threshold = expression_data["log2fold_change"].quantile(0.25)

    # mean_expression = expression_data['log2fold_change'].mean()
    # std_expression = expression_data['log2fold_change'].std()

    # high_threshold = mean_expression + std_expression
    # low_threshold = mean_expression - std_expression

    # Get reaction IDs based on high and low expression thresholds
    high_expression_reactions = reaction_scores_df[
        reaction_scores_df["Score"] >= high_threshold
    ]["ReactionID"]
    low_expression_reactions = reaction_scores_df[
        reaction_scores_df["Score"] <= low_threshold
    ]["ReactionID"]

    print(f"High expressed: {len(high_expression_reactions)}")
    print(f"Low expressed: {len(low_expression_reactions)}")

    print(f"Gene Count in IQR: {reaction_scores_df['ReactionID'].count()}")

    for rxn_id in high_expression_reactions:
        model.reactions.get_by_id(rxn_id).lower_bound = 0

    for rxn_id in low_expression_reactions:
        model.reactions.get_by_id(rxn_id).upper_bound = 0

    return model


integrated_model = apply_imat(base_model, reaction_scores_df, filtered_data)
print("Model integrated with the expression data")

In [ ]:
# Each reaction carried out in steady state conditions
fva_result_base_model = flux_variability_analysis(base_model, fraction_of_optimum=0.9)
fva_result_integrated_model = flux_variability_analysis(
    integrated_model, fraction_of_optimum=0.9
)

# Save the FVA results
fva_result_base_model.to_csv("fva_base_model.csv")
fva_result_integrated_model.to_csv("fva_integrated_model.csv")

In [ ]:
fva_result_base_model = pd.read_csv("fva_base_model.csv")
fva_result_integrated_model = pd.read_csv("fva_integrated_model.csv")

# Rename the first unnamed column
fva_result_base_model_df = fva_result_base_model.rename(
    columns={fva_result_base_model.columns[0]: "ReactionID"}
)
fva_result_integrated_model_df = fva_result_integrated_model.rename(
    columns={fva_result_integrated_model.columns[0]: "ReactionID"}
)
print(fva_result_base_model_df)

In [ ]:
# Combine the results
fva_combined = pd.merge(
    fva_result_base_model_df,
    fva_result_integrated_model_df,
    on="ReactionID",
    suffixes=("_base_model", "_integrated_model"),
)
print(fva_combined)
fva_combined.to_csv("fva_combined.csv", index=False)

In [ ]:
# Calculate the differences
fva_combined["minimum_flux_diff"] = (
    fva_combined["minimum_integrated_model"] - fva_combined["minimum_base_model"]
)
fva_combined["maximum_flux_diff"] = (
    fva_combined["maximum_integrated_model"] - fva_combined["maximum_base_model"]
)
fva_combined

In [ ]:
# Perform t-test
t_stat, p_value = ttest_ind(
    fva_combined["minimum_flux_diff"], fva_combined["maximum_flux_diff"]
)
fva_combined["p_value"] = p_value
fva_combined

In [ ]:
# Plotting the differences
plt.figure(figsize=(12, 8))

# Plot minimum flux differences
plt.subplot(2, 1, 1)
sns.barplot(x=fva_combined.index, y="maximum_base_model", data=fva_combined)
plt.axhline(0, color="gray", linewidth=0.5)
plt.ylabel("Minimum Flux Difference")
plt.title("Minimum Flux Differences between Integrated and Base Models")

# Plot maximum flux differences
plt.subplot(2, 1, 2)
sns.barplot(x=fva_combined.index, y="minimum_integrated_model", data=fva_combined)
plt.axhline(0, color="gray", linewidth=0.5)
plt.ylabel("Maximum Flux Difference")
plt.title("Maximum Flux Differences between Integrated and Base Models")

plt.tight_layout()
plt.show()

In [ ]:
# Calculate -log10(p-value)
pathway_p_values["-log10(p-value)"] = -np.log10(pathway_p_values["p_value"])

# Plot with Seaborn
plt.figure(figsize=(10, 6))
sns.barplot(x="-log10(p-value)", y="pathway", data=pathway_p_values, palette="viridis")
plt.title("-log10(p-value) for Pathways")
plt.xlabel("-log10(p-value)")
plt.ylabel("Pathway")
plt.tight_layout()
plt.show()

#### Add `status` column to the `imat_fva_results.csv` file, to later check which reactions are active and rename the first column to `reaction`


In [ ]:
fva_result = pd.read_csv(
    "/Users/karthik/Desktop/PHCCO IISc Internship/EMT/imat_fva_results.csv"
)
fva_result.head()

In [ ]:
# Rename the first column to 'reaction'
fva_result.rename(columns={fva_result.columns[0]: "reaction"}, inplace=True)

# Initialize all as inactive
fva_result["status"] = "Inactive"

# Update status based on flux bounds
fva_result.loc[(fva_result["minimum"] < 0) & (fva_result["maximum"] > 0), "status"] = (
    "Active"
)
fva_result.loc[
    (fva_result["minimum"] == 0) & (fva_result["maximum"] == 0), "status"
] = "Inactive"
fva_result.loc[
    ((fva_result["minimum"] == 0) & (fva_result["maximum"] < 0))
    | ((fva_result["minimum"] > 0) & (fva_result["maximum"] == 0)),
    "status",
] = "SemiActive"

print(fva_result)

In [ ]:
fva_result.to_csv("imat_fva_results.csv", index=False)

In [ ]:
# Filter reactions based on activity (adjust based on your CSV structure)
active_epithelial = fva_result[fva_result["status"] == "Active"]["reaction"]
active_hybrid = fva_result[fva_result["status"] == "Active"]["reaction"]
active_mesenchymal = fva_result[fva_result["status"] == "Active"]["reaction"]

print("Active Epithelial: \n", active_epithelial)
print("Active Hybrid: \n", active_hybrid)
print("Active Mesenchymal: \n", active_mesenchymal)

In [ ]:
# Create context-specific models
epithelial_model = model.copy()
hybrid_model = model.copy()
mesenchymal_model = model.copy()

# Set bounds based on active reactions
for reaction in epithelial_model.reactions:
    if reaction.id not in active_epithelial:
        reaction.lower_bound = 0.0
        reaction.upper_bound = 0.0

for reaction in hybrid_model.reactions:
    if reaction.id not in active_hybrid:
        reaction.lower_bound = 0.0
        reaction.upper_bound = 0.0

for reaction in mesenchymal_model.reactions:
    if reaction.id not in active_mesenchymal:
        reaction.lower_bound = 0.0
        reaction.upper_bound = 0.0

cobra.io.write_sbml_model(epithelial_model, "epithelial_model.xml")
cobra.io.write_sbml_model(hybrid_model, "hybrid_model.xml")
cobra.io.write_sbml_model(mesenchymal_model, "mesenchymal_model.xml")

### iMAT:


In [ ]:
# Define epithelial and mesenchymal gene sets
epithelial_genes = [
    "CDH1",
    "EPCAM",
    "OCLN",
    "CLDN1",
    "KRT8",
    "KRT18",
    "DSG3",
    "MUC1",
    "JUP",
    "SPINT2",
    "GATA3",
    "GRHL2",
    "SFN",
    "CRB3",
]
mesenchymal_genes = [
    "VIM",
    "FN1",
    "CDH2",
    "SNAI1",
    "SNAI2",
    "TWIST1",
    "ZEB1",
    "ZEB2",
    "MMP2",
    "MMP9",
    "NID1",
    "COL1A1",
    "COL3A1",
    "THY1",
]

### Mapping Genes to Reactions:

GPR rules in the model for mapping genes/proteins to their corresponding reactions.


In [ ]:
# Map genes to reactions
def map_genes_to_reactions(gene_list, expression_dict, model):
    reaction_ids = []
    for reaction in model.reactions:
        for gene in reaction.genes:
            gene_id = gene.id
            if gene_id in gene_list and gene_id in expression_dict:
                reaction_ids.append(reaction.id)
    return list(set(reaction_ids))


epithelial_rxns = map_genes_to_reactions(epithelial_genes, expression_dict, model)
mesenchymal_rxns = map_genes_to_reactions(mesenchymal_genes, expression_dict, model)

### Defining Constraints:

1. **High Expression:** For reactions associated with highly expressed genes, set a lower bound for the flux.
2. **Low Expression:** For reactions associated with lowly expressed genes, set an upper bound for the flux.
3. **Medium Expression:** Reactions associated with medium expression genes are not constrained.


In [ ]:
# Apply constraints
for rxn_id in epithelial_rxns:
    model.reactions.get_by_id(rxn_id).lower_bound = 0.1

for rxn_id in mesenchymal_rxns:
    model.reactions.get_by_id(rxn_id).upper_bound = 0.1

### Optimisation:

**Objective:** Maximize the sum of fluxes through reactions associated with highly expressed genes minus the sum of fluxes through reactions associated with lowly expressed genes.

**Constraints:** flux bounds


In [ ]:
# Define the new objective function
objective_coefficients = {}

for rxn_id in epithelial_rxns:
    objective_coefficients[rxn_id] = 1.0

for rxn_id in mesenchymal_rxns:
    objective_coefficients[rxn_id] = -1.0

# Set the objective
for rxn_id, coefficient in objective_coefficients.items():
    model.reactions.get_by_id(rxn_id).objective_coefficient = coefficient

# Solve the model
solution = pfba(model)
print(f"Objective value: {solution.objective_value}")

# Save the flux distribution
fluxes = solution.fluxes
fluxes.to_csv("imat_fluxes.csv")

### Context Specific Models:

- Using Gene sets specific to epithelial and mesenchymal phenotypes.
- Apply iMAT separately using the specific gene sets for epithelial and mesenchymal models.
